# Public SEC Covenant Monitoring RAG

A project-specific clone of `02-rag.ipynb`. It retrieves public SEC document records for analyst review of covenant terms, reported leverage, compliance events, amendments, and waivers. The pipeline keeps the course structure: `search`, `build_prompt`, `llm`, and `rag`.

In [ ]:
import os
from functools import partial
from pathlib import Path

from credit_monitoring.ingestion.chunking.evidence import chunk_evidence_documents
from credit_monitoring.ingestion.loaders.sec import build_company_catalog, create_documents
from credit_monitoring.retrieval.lexical.sec import build_index

# Explicit filesystem configuration; reusable module imports have no side effects.
data_dir = Path("data")
if not data_dir.exists():
    data_dir = Path("../data")
html_dir = data_dir / "sec_documents" / "html"
markdown_dir = data_dir / "sec_documents" / "markdown"
source_file = data_dir / "public_sec_documents.csv"


In [ ]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()


## 1. Download, clean, convert, and load the selected documents

The loader downloads each HTML filing listed in the selected CSV, saves the original HTML, removes clear page chrome with Beautiful Soup, and converts the cleaned HTML to Markdown. It then selects covenant-related evidence lines with neighboring context and indexes the resulting Markdown chunks with the catalog metadata.

SEC requests need an identifying `SEC_USER_AGENT` environment variable containing the requester's name or organization and a real contact email. Cached HTML and Markdown files can be reused. The first run after adding HTML cleanup reconverts cached HTML so older Markdown does not bypass cleanup.


In [ ]:
# Downloads use your configured identity; cached filings need no network request.
sec_user_agent = os.environ.get("SEC_USER_AGENT", "")


In [ ]:
from markitdown import MarkItDown

converter = MarkItDown(enable_plugins=False)


## 2. Build the Index

Index covenant-related evidence excerpts with issuer and catalog metadata. Each result retains the filing citation and Markdown path for verification.


In [ ]:
filing_documents = create_documents(
    source_file,
    html_dir=html_dir,
    markdown_dir=markdown_dir,
    user_agent=sec_user_agent,
    converter=converter,
)
documents = chunk_evidence_documents(filing_documents, size=3500, step=500)
index = build_index(documents)
print(f"Loaded {len(filing_documents)} filings and {len(documents)} evidence chunks.")
print(f"Original HTML: {html_dir} | Markdown: {markdown_dir}")


In [ ]:
documents[0] if documents else "No evidence excerpts were selected."


## 3. Import and bind the RAG functions

Reusable functions live in `src/credit_monitoring`. Bind the explicit client and index here; prompts are imported from dedicated modules. Narrative responses report source-stated facts without calculating ratios or headroom.


In [ ]:
from credit_monitoring.retrieval.lexical.sec import search as search_records

search = partial(search_records, index=index, document_count=len(documents))


In [ ]:
from credit_monitoring.agents.prompts.assembly import build_prompt
from credit_monitoring.agents.prompts.narrative import NARRATIVE_INSTRUCTIONS as instructions


In [ ]:
from credit_monitoring.application.rag_service import llm as generate_narrative

llm = partial(generate_narrative, client=openai_client)


In [ ]:
from credit_monitoring.application.rag_service import rag as run_rag

rag = partial(run_rag, client=openai_client, index=index, document_count=len(documents))


In [ ]:
search_results = search( """
            For FMC Corporation (FMC), summarize the reported net leverage trend and covenant status in the available SEC records. 
            Cite each record and explain what evidence is still needed to assess current covenant headroom.
        """)

print(search_results)

## 4. Try It Out

Both prompts search the Markdown evidence chunks and include up to two relevant chunks per filing in the LLM context.


### Prompt 1: FMC borrower review


In [ ]:
print(
    rag(
        """
            For FMC Corporation (FMC), summarize the reported net leverage trend and covenant status in the available SEC records. 
            Cite each record and explain what evidence is still needed to assess current covenant headroom.
        """
    )
)


### Prompt 2: Full-corpus comparison table


In [ ]:
print(
    rag(
        """
            Across all indexed SEC filings, compare each issuer's latest disclosed leverage covenant result or status with 
            available prior periods. Present the findings in a concise table for quick analyst review, with columns for issuer, 
            latest period, leverage result, covenant threshold, compliance or waiver status, comparison with prior period, 
            and source citation. Include applicable thresholds and relevant amendments or waivers where the filings support them. 
            Mark unavailable information as "Not disclosed in retrieved sources," and do not infer missing values or status.
        """
    )
)

## 5. Structured covenant extraction V2

The single-company extractor uses retrieved results. Each result contains `extraction`, `validation`, and schema/prompt versions. Validation errors retain the unchanged facts for review. Refusals, incomplete responses, and parsing failures raise typed extraction errors.

Missing fields stay null and gaps describe only the supplied excerpts. The extractor does not choose governing amendments, calculate covenant values, or infer compliance. Chunk offsets refer to keyword-selected evidence text, not the original Markdown file.


In [ ]:
from credit_monitoring.domain import CovenantExtraction
from credit_monitoring.agents.prompts.extraction import EXTRACTION_INSTRUCTIONS
from credit_monitoring.covenants.extraction import extract_covenants as extract_records

extract_covenants = partial(extract_records, client=openai_client)


In [ ]:
# Example: retrieve evidence first, then produce a typed result that can be saved or reviewed.
extraction_query = """ 
    For FMC Corporation (FMC), extract disclosed leverage covenant terms,
    reported results, testing periods, and any amendments or waivers. 
    Identify what is missing to establish the active covenant.
 """

In [ ]:
fmc_evidence = search(extraction_query)
print(fmc_evidence)

In [ ]:
fmc_extraction = extract_covenants(extraction_query, fmc_evidence)

In [ ]:
# The serialized result contains extraction, schema/prompt versions, and validation.
# Inspect validation.is_valid and issues before using any extracted facts.
print(fmc_extraction.model_dump_json(indent=2))
print("Evidence validation passed:", fmc_extraction.validation.is_valid)


### Extract covenant data for all companies

One model request covers every catalog company. Each `companies` entry contains `company`, `ticker`, and a complete V2 `extraction`. The result report includes coverage and company-specific evidence checks.


In [ ]:
from credit_monitoring.domain import AllCompanyCovenantExtraction
from credit_monitoring.agents.prompts.batch import ALL_COMPANY_INSTRUCTIONS
from credit_monitoring.covenants.extraction import extract_all_companies as extract_catalog


In [ ]:
company_by_ticker = build_company_catalog(filing_documents)
extract_all_companies = partial(
    extract_catalog,
    documents=documents,
    company_by_ticker=company_by_ticker,
    client=openai_client,
)


In [ ]:

# One model call covers every company and every indexed covenant evidence excerpt.
all_company_query = """ 
    For every company in the SEC catalog, extract disclosed covenant names, metrics, thresholds, testing periods, 
    reported results, compliance status, and amendments or waivers. Summarize evidence gaps and conflicts. 
    
    Do not infer missing terms or calculate headroom.
"""

In [ ]:
all_company_extraction = extract_all_companies(all_company_query)
print(all_company_extraction.model_dump_json(indent=2))
print("Evidence validation passed:", all_company_extraction.validation.is_valid)
